In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import numpy as np
from sklearn.model_selection import train_test_split
scaler = StandardScaler()
imputer = SimpleImputer(strategy = 'median')

#A lot of stuff was commented out. only trust things up until ML_df and train_test_split. i previously tried to normalize the values before splittign which is a bad idea
# that would result in data leakage.
#ML_df has around 340 features which might be pretty bad. i think best next move is to normalize the players performance in the combine things and combine those somehow


In [2]:
#read player

player_crr_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/player_career_successes.csv')
player_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/players.csv')
# player_ply_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/player_play.csv')
print('read players')


read players


In [3]:
# read games

games_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/games.csv')
print('read games')

read games


In [4]:
#read games 2023
game_trk_2023_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/game_tracking_2023.csv')
print('read games 2023')


read games 2023


In [5]:
#read games 2024
game_trk_2024_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/game_tracking_2024.csv')
print('read games 2024')


read games 2024


In [6]:
#read games 2025
game_trk_2025_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/game_tracking_2025.csv')
print('read games 2025')

read games 2025


In [7]:
#read combine_res
combine_res_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/combine_results.csv')
print("read combine_res")
mapToOffense = {"WR": "Offense", "TE": "Offense", "OL": "Offense", "DL": "Defense", "DB": "Defense"}

read combine_res


In [8]:
#read combine_trk
combine_trk_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/combine_tracking.csv')
print("read combine_trk")

read combine_trk


In [9]:
#List of all current dfs
dfs = {'combine_res': combine_res_df, 'game_trk_2023': game_trk_2023_df, 'game_trk_2024': game_trk_2024_df, 
       'game_trk_2025': game_trk_2025_df, 'player_crr': player_crr_df, 'player': player_df, 
       'combine_trk': combine_trk_df, 'games': games_df}

#combine_res_df gives their positions and stats. these stats should be compared to others in same position. each position only appears once
#game_trk_202*_df gives speed info... still get their means and max/min from here use for later
#combine_trk_df gives drill type and drill name. separate for now and create a df where all their same drill names are grouped
#im going to ignore x, y, dir, attempt, (maybe dis), draft_year for now in combine_trk_df 
game_trk_2023_df.head()

,game_id,play_id,nfl_id,time,x,y,s,a,dis,o,dir,event
0,2024010707,570,56450,2024-01-07T21:46:07.700,42.17,25.29,0.0,0.0,0.0,290.97,213.26,NaN
1,2024010707,570,56450,2024-01-07T21:46:07.800,42.17,25.29,0.0,0.0,0.0,290.97,210.36,NaN
2,2024010707,570,56450,2024-01-07T21:46:07.900,42.18,25.29,0.0,0.0,0.0,290.97,205.45,NaN
3,2024010707,570,56450,2024-01-07T21:46:08.000,42.18,25.29,0.0,0.0,0.0,290.97,200.73,NaN
4,2024010707,570,56450,2024-01-07T21:46:08.100,42.18,25.29,0.0,0.0,0.0,290.97,199.27,NaN


In [10]:
#player_crr
player_crr_df["career_snaps"] = player_crr_df[
    ["career_offensive_snaps", "career_defensive_snaps"]
].sum(axis=1)

player_crr_df = player_crr_df.drop(
    columns=["career_offensive_snaps", "career_defensive_snaps"]
)

In [11]:
#=========================================== Begin combine_trk_df normalization======================
combine_trk_df = combine_trk_df[
    combine_trk_df['entity_type'] == 'PLAYER'
    ]

In [12]:
combine_trk_df = combine_trk_df.drop(
    columns = ['x', 'y', 'dis', 'dir', 'entity_type']
)

In [13]:

combine_trk_df = combine_trk_df.groupby(
    ['drill_type', 'drill_name', 'nfl_id', 'draft_year']).agg(
    max_speed_combinetrk=('s','max'),
    avg_speed_combinetrk=('s', 'mean'),
    max_acceleration_combinetrk=('a','max'),
    avg_acceleration_combinetrk=('a','mean'),
    min_acceleration_combinetrk=('a','min'),
    total_frames_combinetrk = ('s','count')
).reset_index()

In [14]:
metrics = [
    'max_speed_combinetrk',
    'avg_speed_combinetrk',
    'max_acceleration_combinetrk',
    'avg_acceleration_combinetrk',
    'min_acceleration_combinetrk'
]

combine_trk_df = combine_trk_df.pivot(
    index=['nfl_id', 'draft_year'],
    columns=['drill_type', 'drill_name'],
    values=metrics
)

combine_trk_df.columns = [
    f'{metric}_{drill_type}_{drill_name}'
    for metric, drill_type, drill_name in combine_trk_df.columns
]

combine_trk_df = combine_trk_df.reset_index()

#many nan values now. ok for our case but for data bowl should be handled

In [15]:
scale_columns = ['max_speed', 'avg_speed', 
                 'max_acceleration', 'avg_acceleration', 
                 'min_acceleration']
#after split
# for col in scale_columns:
#     combine_trk_df[col] = combine_trk_df.groupby(['drill_type', 'drill_name', 'draft_year'])[col].transform(
#         lambda x: scaler.fit_transform(x.values.reshape(-1, 1)).flatten()
#     )

# all rows are normalized for their specific drill type and drill name. (at least should be...)

#=========================================== End combine_trk_df normalization======================

In [16]:
#player_crr_df ready to split
#combine_trk_df ready to split
#combine_res ready to split

In [17]:

#=========================================norm combine_res_dfs======================================
# combine_res_dfs_sep = {
#     value: df for value, df in dfs['combine_res'].groupby('combine_position')
# }

In [18]:
#imputer for combine_res

# for key, val in combine_res_dfs_sep.items():
#     cols_with_nan = val.columns[
#         val.isna().any()
#     ].tolist()
    
#     val[cols_with_nan] = imputer.fit_transform(
#         val[cols_with_nan]
#     )
#     combine_res_dfs_sep[key] = val

#removed all nans too prior to z scores with imputation (replace with median)

In [19]:
# merged_combine_res_df = pd.concat(
#     combine_res_dfs_sep, axis=0, ignore_index=True
# )

In [20]:
# merged_combine_res_df.head()

In [21]:



# scale_columns = ['combine_height', 'combine_weight', 'hand_size', 
#                  'arm_length', 'wing_span', 'ten_yd_split', 'forty', 
#                  'vertical', 'broad_jump', 'three_cone', 'short_shuttle', 'bench_reps', 
#                  'ngs_athleticism_score', 'ngs_college_production_score', 'ngs_final_score']

# ct = ColumnTransformer(
#         transformers=[
#             ('num_scaler', StandardScaler(), scale_columns)
#         ],
#         remainder='passthrough'
#     )
# ct.set_output(transform='pandas')

# for key, val in combine_res_dfs_sep.items():
#     val[scale_columns] = scaler.fit_transform(
#         val[scale_columns]
#     )
#     combine_res_dfs_sep[key] = val

#now all have z scores dependent on their position 

In [22]:
#after split
# merged_aggr_combine_res_df = pd.concat(
#     combine_res_dfs_sep, axis=0, ignore_index=True
# )

#concat/merge all combine dfs with norm vals

In [23]:
# merged_aggr_combine_res_df.head()
#===========================================end norm combine_res_dfs====================================

In [24]:

#==========================================================Begin Cleaning game_[year] ===============================================================
dfs['game_trk_2025'] = dfs['game_trk_2025'][dfs['game_trk_2025']['s'] > 0]
aggr_games_trk2025_df = dfs['game_trk_2025'].groupby(
    ['nfl_id']).agg(
    max_speed=('s','max'),
    avg_speed=('s', 'mean'),
    max_acceleration=('a','max'),
    avg_acceleration=('a','mean'),
    total_frames=('s','count')
).reset_index()


In [25]:
dfs['game_trk_2024'] = dfs['game_trk_2024'][dfs['game_trk_2024']['s'] > 0]
aggr_games_trk2024_df = dfs['game_trk_2024'].groupby(
    ['nfl_id']).agg(
    max_speed=('s','max'),
    avg_speed=('s', 'mean'),
    max_acceleration=('a','max'),
    avg_acceleration=('a','mean'),
    total_frames=('s','count')
).reset_index()


In [26]:
dfs['game_trk_2023'] = dfs['game_trk_2023'][dfs['game_trk_2023']['s'] > 0]
aggr_games_trk2023_df = dfs['game_trk_2023'].groupby(
    ['nfl_id']).agg(
    max_speed=('s','max'),
    avg_speed=('s', 'mean'),
    max_acceleration=('a','max'),
    avg_acceleration=('a','mean'),
    total_frames=('s','count')
).reset_index()

                  

In [27]:
all_aggr_games_trk_df = pd.concat(
    [aggr_games_trk2023_df, aggr_games_trk2024_df, aggr_games_trk2025_df], 
    ignore_index = True
)
# all_aggr_games_trk_df = all_aggr_games_trk_df.groupby('nfl_id', as_index=False).agg(
#     max_speed = ('max_speed', 'max'),
#     avg_speed=('avg_speed', 'mean'),
#     max_acceleration=('max_acceleration','max'),
#     avg_acceleration=('avg_acceleration','mean'),
#     total_frames=('total_frames','sum')
# ).reset_index()

scale_columns = ['max_speed', 'avg_speed', 
                 'max_acceleration', 'avg_acceleration']

# all_aggr_games_trk_df[scale_columns] = scaler.fit_transform(
#     all_aggr_games_trk_df[scale_columns]
# )

#=============================================================================END cleaning game_[year]=================================================

In [28]:
#==================================================== begin clean player csv================================
player_df['Age'] = ((pd.to_datetime(player_df['draft_year'], format='%Y') - pd.to_datetime(player_df['birth_date'])).dt.days / 365.25).astype(int)
player_df = player_df.drop(columns = ['birth_date', 'draft_year'])


In [29]:
#using all game_trk, combine_trk, combine_res, player, player_crr_df
ML_df = pd.merge(all_aggr_games_trk_df, combine_trk_df, on = 'nfl_id')
ML_df = pd.merge(ML_df, combine_res_df, on = 'nfl_id')
ML_df = pd.merge(ML_df, player_df, on = 'nfl_id')
ML_df = pd.merge(ML_df, player_crr_df, on = 'nfl_id')


In [38]:
print(ML_df.columns)

Index(['nfl_id', 'max_speed', 'avg_speed', 'max_acceleration',
       'avg_acceleration', 'total_frames', 'draft_year_x',
       'max_speed_combinetrk_FORTY_YARD_DASH_FORTY_YARD_DASH',
       'max_speed_combinetrk_SHORT_SHUTTLE_SHORT_SHUTTLE',
       'max_speed_combinetrk_SKILL_DRILLS_DB_BACK_PEDAL_AND_90_DEGREE_BREAK_TO_BALL',
       ...
       'draft_pick_within_round', 'draft_overall_pick', 'Age',
       'career_special_teams_snaps', 'career_games_active',
       'career_games_started', 'ap_all_pro_1st_team', 'ap_all_pro_2nd_team',
       'pro_bowl_original_ballot', 'career_snaps'],
      dtype='str', length=349)


In [39]:
X = ML_df.drop(columns = 'draft_overall_pick')
y = ML_df['draft_overall_pick']

In [40]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

In [46]:
# pd.reset_option('display.max_columns')
X_train.head()

,nfl_id,max_speed,avg_speed,max_acceleration,avg_acceleration,total_frames,draft_year_x,max_speed_combinetrk_FORTY_YARD_DASH_FORTY_YARD_DASH,max_speed_combinetrk_SHORT_SHUTTLE_SHORT_SHUTTLE,max_speed_combinetrk_SKILL_DRILLS_DB_BACK_PEDAL_AND_90_DEGREE_BREAK_TO_BALL,...,draft_round,draft_pick_within_round,Age,career_special_teams_snaps,career_games_active,career_games_started,ap_all_pro_1st_team,ap_all_pro_2nd_team,pro_bowl_original_ballot,career_snaps
457,57606,10.34,4.136822,8.52,2.460503,22304,2024,10.716667,NaN,NaN,...,NaN,NaN,22,32,25,7,0,0,0,869
799,58298,12.83,3.025488,16.55,1.927941,79342,2025,11.134096,NaN,8.455991,...,3.0,29.0,22,100,17,16,0,0,0,977
25,55918,8.22,2.281323,6.32,1.435631,36925,2023,9.970000,NaN,NaN,...,2.0,22.0,22,242,51,33,0,0,0,1713
84,56042,11.78,3.738445,11.17,2.486497,40392,2023,NaN,NaN,NaN,...,5.0,43.0,21,11,46,43,1,1,2,2211
10,55888,10.39,2.744476,8.93,1.830879,63899,2023,11.200000,NaN,7.213333,...,1.0,25.0,21,217,51,35,0,0,0,2091


In [31]:
#checkpoint before encoding
#combine_res_df -> merged_aggr_combine_res_df
#game_trk_2025_df, game_trk_2024_df, game_trk_2023_df -> all_aggr_games_trk_df
#combine_trk_df -> combine_trk_df (but cleaned)
#all left is player, player_crr, and game_df

# progress_so_far = [merged_aggr_combine_res_df, all_aggr_games_trk_df, combine_trk_df]

#grab age of player, college, conference, 
#draft_round, draft_pick_within_round, draft_overall_pick from player_df

#grab career_offensive_snaps, career_defensive_snaps, career_special_team_snaps, career_games_active, career_games_started, ap_all_pro_1st_team, ap_all_pro_2nd_team pro_bowl_original_ballot

In [14]:
#======================================================================Begin encoding==========================================
aggr_combine_trk_df = pd.get_dummies(aggr_combine_trk_df, columns = ['drill_type'], dtype=int)

In [15]:
defensive_pos = ["Defensive Tackle",
  "Nose Tackle",
  "Defensive End",
  "Edge Rusher",
  "Middle Linebacker",
  "Inside Linebacker",
  "Outside Linebacker",
  "Cornerback",
  "Free Safety",
  "Strong Safety",
  "Nickelback",
  "Dimeback"]

offensive_pos = ["Quarterback",
  "Running Back",
  "Fullback",
  "Wide Receiver",
  "Tight End",
  "Center",
  "Left Guard",
  "Right Guard",
  "Left Tackle",
  "Right Tackle",
  "Offensive Guard",
  "Offensive Tackle"]

player_df = pd.get_dummies(player_df, columns = ['nfl_position', 'college_name', 'college_conference'], dtype=int)

In [19]:
combine_res_df = pd.get_dummies(combine_res_df, columns = ['combine_position'], dtype=int)
#===============================================================end encoding =========================================================================

In [20]:

#====================================================Begin Standardizing ===============================

encoded_dfs = {'aggr_combine_trk': aggr_combine_trk_df, 'all_aggr_games_trk_df': all_aggr_games_trk_df, 
             'player_crr': player_crr_df, 'player': player_df, 'combine_res': combine_res_df}

scaler = StandardScaler()
aggr_combine_trk_df.head()

,nfl_id,max_speed,avg_speed,max_acceleration,avg_acceleration,min_acceleration,total_frames,drill_type_FORTY_YARD_DASH,drill_type_SHORT_SHUTTLE,drill_type_SKILL_DRILLS_DB,drill_type_SKILL_DRILLS_DL,drill_type_SKILL_DRILLS_LB,drill_type_SKILL_DRILLS_OL,drill_type_SKILL_DRILLS_TE,drill_type_SKILL_DRILLS_WR,drill_type_THREE_CONE_DRILL
0,55867,10.416667,7.966603,5.859557,1.921288,0.076739,52,1,0,0,0,0,0,0,0,0
1,55867,8.630000,4.453024,8.137175,2.873069,0.014907,797,0,0,0,0,1,0,0,0,0
2,55870,8.213333,3.117726,6.552510,2.920588,0.096148,557,0,0,0,0,0,1,0,0,0
3,55874,10.073333,7.554877,5.126226,1.827326,0.161383,54,1,0,0,0,0,0,0,0,0
4,55874,7.876667,2.914891,6.696734,2.675264,0.080069,597,0,0,0,0,0,1,0,0,0


In [21]:
columns_to_transform = aggr_combine_trk_df.columns.difference(['nfl_id'])
aggr_combine_trk_df[columns_to_transform] = scaler.fit_transform(
    aggr_combine_trk_df[columns_to_transform]
)

In [22]:
columns_to_transform = all_aggr_games_trk_df.columns.difference(['nfl_id'])
all_aggr_games_trk_df[columns_to_transform] = scaler.fit_transform(
    all_aggr_games_trk_df[columns_to_transform]
)

In [23]:
all_aggr_games_trk_df = all_aggr_games_trk_df.drop(columns = 'index')

In [24]:
player_crr_df.head()
#separate by position

,nfl_id,career_offensive_snaps,career_defensive_snaps,career_special_teams_snaps,career_games_active,career_games_started,ap_all_pro_1st_team,ap_all_pro_2nd_team,pro_bowl_original_ballot
0,55867,0,1884,92,51,44,1,0,1
1,55870,2802,0,169,47,43,0,0,0
2,55874,3222,0,167,51,49,0,1,0
3,55875,3011,0,170,51,48,0,0,0
4,55877,0,1056,387,51,2,0,0,0
